<a href="https://www.kaggle.com/code/helenylmaz/smartphone-addiction-prediction?scriptVersionId=345931848" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/playground-series-s6e8/sample_submission.csv
/kaggle/input/competitions/playground-series-s6e8/train.csv
/kaggle/input/competitions/playground-series-s6e8/test.csv


In [2]:
import pandas as pd 
import numpy as np
train=pd.read_csv('/kaggle/input/competitions/playground-series-s6e8/train.csv')
test=pd.read_csv('/kaggle/input/competitions/playground-series-s6e8/test.csv')
#veri boyutuna bakalım elimizde neler varmış
print("train boyutu :",train.shape)
print("test boyutu :",test.shape)
#ilk 5 satırı göstersin
train.head()

train boyutu : (691369, 14)
test boyutu : (296302, 13)


,id,age,daily_screen_time_hours,social_media_hours,gaming_hours,work_study_hours,sleep_hours,notifications_per_day,app_opens_per_day,weekend_screen_time,gender,stress_level,academic_work_impact,addicted_label
0,0,24.0,NaN,1.83,1.59,2.11,7.46,122.0,38.0,8.63,Male,Medium,No,1
1,1,19.0,5.97,1.08,NaN,3.03,8.22,76.0,19.0,NaN,Female,Medium,No,0
2,2,18.0,5.09,NaN,NaN,NaN,6.25,134.0,60.0,7.47,Female,Low,Yes,0
3,3,21.0,6.42,1.26,1.42,3.36,8.85,112.0,94.0,8.66,Other,Low,NaN,1
4,4,26.0,11.20,1.87,2.81,1.95,5.25,NaN,NaN,13.39,Female,Medium,No,1


In [3]:
train.info()
train.isnull().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 691369 entries, 0 to 691368
Data columns (total 14 columns):
 #   Column                   Non-Null Count   Dtype  
---  ------                   --------------   -----  
 0   id                       691369 non-null  int64  
 1   age                      662440 non-null  float64
 2   daily_screen_time_hours  595515 non-null  float64
 3   social_media_hours       557374 non-null  float64
 4   gaming_hours             564548 non-null  float64
 5   work_study_hours         639851 non-null  float64
 6   sleep_hours              646889 non-null  float64
 7   notifications_per_day    623785 non-null  float64
 8   app_opens_per_day        610659 non-null  float64
 9   weekend_screen_time      579306 non-null  float64
 10  gender                   662335 non-null  object 
 11  stress_level             636221 non-null  object 
 12  academic_work_impact     647145 non-null  object 
 13  addicted_label           691369 non-null  int64  
dtypes: f

id                              0
age                         28929
daily_screen_time_hours     95854
social_media_hours         133995
gaming_hours               126821
work_study_hours            51518
sleep_hours                 44480
notifications_per_day       67584
app_opens_per_day           80710
weekend_screen_time        112063
gender                      29034
stress_level                55148
academic_work_impact        44224
addicted_label                  0
dtype: int64

In [4]:
missing_df = pd.DataFrame({
    'missing_count': train.isnull().sum(),
    'missing_pct': (train.isnull().sum() / len(train)) * 100
}).sort_values(by='missing_pct', ascending=False)

print(missing_df)

                         missing_count  missing_pct
social_media_hours              133995    19.381112
gaming_hours                    126821    18.343461
weekend_screen_time             112063    16.208855
daily_screen_time_hours          95854    13.864376
app_opens_per_day                80710    11.673940
notifications_per_day            67584     9.775388
stress_level                     55148     7.976638
work_study_hours                 51518     7.451592
sleep_hours                      44480     6.433612
academic_work_impact             44224     6.396584
gender                           29034     4.199494
age                              28929     4.184307
id                                   0     0.000000
addicted_label                       0     0.000000


In [5]:
# 3 alt kategorinin toplamı
sub_total = train['social_media_hours'].fillna(0) + train['gaming_hours'].fillna(0) + train['work_study_hours'].fillna(0)

# Mantıksal tutarsızlık olan satır sayısı (Alt kategoriler toplamı > Günlük Ekran Süresi)
inconsistent_rows = train[sub_total > train['daily_screen_time_hours']]

print("Mantıksal tutarsızlık gösteren satır sayısı:", len(inconsistent_rows))

Mantıksal tutarsızlık gösteren satır sayısı: 0


In [6]:
# 1. Kategorik sütunlardaki benzersiz değerleri ve eksikleri görelim
for col in ['gender', 'stress_level', 'academic_work_impact']:
    print(f"--- {col} ---")
    print(train[col].value_counts(dropna=False))
    print("\n")

# 2. Sayısal sütunların genel özet istatistikleri (Ortalama, Standart Sapma, Min, Max)
train.describe().T

--- gender ---
gender
Male      223662
Female    221595
Other     217078
NaN        29034
Name: count, dtype: int64


--- stress_level ---
stress_level
High      220873
Low       207783
Medium    207565
NaN        55148
Name: count, dtype: int64


--- academic_work_impact ---
academic_work_impact
Yes    330566
No     316579
NaN     44224
Name: count, dtype: int64




,count,mean,std,min,25%,50%,75%,max
id,691369.0,345684.000000,199581.183466,0.00,172842.00,345684.00,518526.00,691368.00
age,662440.0,26.615408,5.153162,18.00,22.00,27.00,31.00,35.00
daily_screen_time_hours,595515.0,7.640865,2.721446,0.50,5.48,7.77,9.84,15.00
social_media_hours,557374.0,2.471038,1.316137,0.00,1.45,2.31,3.37,8.00
gaming_hours,564548.0,1.459265,0.934552,0.00,0.70,1.33,2.09,4.00
work_study_hours,639851.0,2.366971,1.258797,0.00,1.36,2.20,3.20,6.00
sleep_hours,646889.0,6.804334,1.234512,4.50,5.78,6.80,7.87,9.00
notifications_per_day,623785.0,145.894900,65.917556,20.00,93.00,150.00,204.00,250.00
app_opens_per_day,610659.0,102.636781,48.093970,15.00,64.00,104.00,145.00,180.00
weekend_screen_time,579306.0,9.479866,2.856006,0.51,7.28,9.58,11.75,17.56


In [7]:
# Verimizin orijinal halini korumak için bir kopya oluşturalım
df = train.copy()

# 1. Ordinal Encoding: stress_level (Sıralı Değişken)
stress_mapping = {'Low': 0, 'Medium': 1, 'High': 2}
df['stress_level_encoded'] = df['stress_level'].map(stress_mapping)

# 2. One-Hot Encoding: gender ve academic_work_impact (Sırasız Değişkenler)
df = pd.get_dummies(df, columns=['gender', 'academic_work_impact'], dummy_na=False, drop_first=True)

# İlk 5 satıra bakalım, yeni sütunlarımız oluşmuş mu?
df.head()

,id,age,daily_screen_time_hours,social_media_hours,gaming_hours,work_study_hours,sleep_hours,notifications_per_day,app_opens_per_day,weekend_screen_time,stress_level,addicted_label,stress_level_encoded,gender_Male,gender_Other,academic_work_impact_Yes
0,0,24.0,NaN,1.83,1.59,2.11,7.46,122.0,38.0,8.63,Medium,1,1.0,True,False,False
1,1,19.0,5.97,1.08,NaN,3.03,8.22,76.0,19.0,NaN,Medium,0,1.0,False,False,False
2,2,18.0,5.09,NaN,NaN,NaN,6.25,134.0,60.0,7.47,Low,0,0.0,False,False,True
3,3,21.0,6.42,1.26,1.42,3.36,8.85,112.0,94.0,8.66,Low,1,0.0,False,True,False
4,4,26.0,11.20,1.87,2.81,1.95,5.25,NaN,NaN,13.39,Medium,1,1.0,False,False,False


In [8]:
# 1. HAMLE: Missing Indicator (Eksiklik Bayrakları) Yaratma
missing_cols = ['social_media_hours', 'gaming_hours', 'daily_screen_time_hours', 'weekend_screen_time']
for col in missing_cols:
    df[f'{col}_is_missing'] = df[col].isnull().astype(int)

# 2. HAMLE: Yaş Grubu (Age Group) Oluşturma (Gruplanmış medyan ve feature'lar için)
# 18-22 (Genç Öğrenci), 23-27 (Üniversite/Yeni Mezun), 28-35 (Çalışan/Yetişkin)
df['age_group'] = pd.cut(df['age'], bins=[17, 22, 27, 35], labels=['Young', 'Mid', 'Adult'])

# 3. HAMLE: Gruplanmış Medyan ile Eksik Veri Doldurma
# Yaş grubu ve cinsiyete göre gruplayıp medyanları basıyoruz
numeric_missing_cols = [
    'daily_screen_time_hours', 'social_media_hours', 'gaming_hours', 
    'work_study_hours', 'sleep_hours', 'notifications_per_day', 
    'app_opens_per_day', 'weekend_screen_time'
]

for col in numeric_missing_cols:
    df[col] = df.groupby(['age_group', 'gender_Male', 'gender_Other'])[col].transform(lambda x: x.fillna(x.median()))

# 4. HAMLE: Domain Feature Engineering (Uzman Özellik Türetimi)
# Sıfıra bölünme hatası (ZeroDivisionError) olmaması için paydalara çok küçük bir epsilon (1e-5) ekliyoruz
df['social_media_ratio'] = df['social_media_hours'] / (df['daily_screen_time_hours'] + 1e-5)
df['gaming_ratio'] = df['gaming_hours'] / (df['daily_screen_time_hours'] + 1e-5)
df['work_study_ratio'] = df['work_study_hours'] / (df['daily_screen_time_hours'] + 1e-5)

df['notification_per_open'] = df['notifications_per_day'] / (df['app_opens_per_day'] + 1e-5)
df['weekend_surge_ratio'] = df['weekend_screen_time'] / (df['daily_screen_time_hours'] + 1e-5)

# Yeni sütun sayımıza ve ilk 5 satıra bakalım
print("Yeni Veri Seti Boyutu (Sütun Sayısı):", df.shape[1])
df.head()

/tmp/ipykernel_16/1941391546.py:19: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df[col] = df.groupby(['age_group', 'gender_Male', 'gender_Other'])[col].transform(lambda x: x.fillna(x.median()))


Yeni Veri Seti Boyutu (Sütun Sayısı): 26


,id,age,daily_screen_time_hours,social_media_hours,gaming_hours,work_study_hours,sleep_hours,notifications_per_day,app_opens_per_day,weekend_screen_time,...,social_media_hours_is_missing,gaming_hours_is_missing,daily_screen_time_hours_is_missing,weekend_screen_time_is_missing,age_group,social_media_ratio,gaming_ratio,work_study_ratio,notification_per_open,weekend_surge_ratio
0,0,24.0,7.86,1.83,1.59,2.11,7.46,122.0,38.0,8.63,...,0,0,1,0,Mid,0.232824,0.202290,0.268447,3.210525,1.097963
1,1,19.0,5.97,1.08,1.32,3.03,8.22,76.0,19.0,9.44,...,0,1,0,1,Young,0.180904,0.221105,0.507537,3.999998,1.581237
2,2,18.0,5.09,2.27,1.32,2.21,6.25,134.0,60.0,7.47,...,1,1,0,0,Young,0.445972,0.259332,0.434184,2.233333,1.467581
3,3,21.0,6.42,1.26,1.42,3.36,8.85,112.0,94.0,8.66,...,0,0,0,0,Young,0.196261,0.221183,0.523364,1.191489,1.348908
4,4,26.0,11.20,1.87,2.81,1.95,5.25,146.0,104.0,13.39,...,0,0,0,0,Mid,0.166964,0.250893,0.174107,1.403846,1.195535


In [9]:
# Aykırı Değer Baskılama (Capping / Winsorization) Fonksiyonu
def replace_outliers_iqr(dataframe, columns):
    df_capped = dataframe.copy()
    for col in columns:
        Q1 = df_capped[col].quantile(0.25)
        Q3 = df_capped[col].quantile(0.75)
        IQR = Q3 - Q1
        
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR
        
        # Aykırı değerleri sınırlara baskılıyoruz
        df_capped[col] = df_capped[col].clip(lower=lower_bound, upper=upper_bound)
    return df_capped

# Aykırı değer kontrolü yapacağımız sayısal sütunlar
numeric_cols = [
    'daily_screen_time_hours', 'social_media_hours', 'gaming_hours',
    'work_study_hours', 'sleep_hours', 'notifications_per_day', 
    'app_opens_per_day', 'weekend_screen_time'
]

# Baskılama işlemini uygulayalım
df = replace_outliers_iqr(df, numeric_cols)

print("Aykırı değerler IQR yöntemiyle başarıyla baskılandı!")
df.describe().T[['min', '25%', '50%', '75%', 'max']]

Aykırı değerler IQR yöntemiyle başarıyla baskılandı!


,min,25%,50%,75%,max
id,0.000000,172842.000000,345684.000000,518526.000000,691368.000000
age,18.000000,22.000000,27.000000,31.000000,35.000000
daily_screen_time_hours,0.500000,5.830000,7.770000,9.570000,15.000000
social_media_hours,0.000000,1.650000,2.310000,3.110000,5.300000
gaming_hours,0.000000,0.830000,1.330000,1.900000,3.505000
work_study_hours,0.000000,1.420000,2.200000,3.110000,5.645000
sleep_hours,4.500000,5.860000,6.810000,7.790000,9.000000
notifications_per_day,20.000000,102.000000,150.000000,197.000000,250.000000
app_opens_per_day,15.000000,68.000000,104.000000,139.000000,180.000000
weekend_screen_time,2.265000,7.710000,9.580000,11.340000,16.785000


In [10]:
from sklearn.model_selection import train_test_split
import lightgbm as lgb
from sklearn.metrics import roc_auc_score, f1_score, classification_report

# 1. Özellikler (X) ve Hedef Değişken (y) Ayrımı
# Modelin girmesini istemediğimiz 'id', 'addicted_label' ve metin olan 'age_group', 'stress_level' sütunlarını çıkarıyoruz
X = df.drop(columns=['id', 'addicted_label', 'age_group', 'stress_level'])
y = df['addicted_label']

# 2. Veriyi Eğitim (%80) ve Doğrulama (%20) Olarak Bölme
X_train, X_val, y_train, y_val = train_test_split(
    X, y, 
    test_size=0.20, 
    random_state=42, 
    stratify=y
)

print(f"Eğitim Seti Boyutu: {X_train.shape}")
print(f"Doğrulama (Val) Seti Boyutu: {X_val.shape}")

# 3. Model Kurma ve Eğitme (LightGBM)
model = lgb.LGBMClassifier(
    n_estimators=500,
    learning_rate=0.05,
    random_state=42,
    n_jobs=-1
)

model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)]
)

# 4. Doğrulama Seti Üzerinde Tahmin Alma
y_pred_proba = model.predict_proba(X_val)[:, 1] # ROC-AUC için olasılık değerleri
y_pred = model.predict(X_val) # F1 ve Accuracy için 0 veya 1 tahminleri

# 5. Skorları Basma
print("\n--- MODEL PERFORMANS SONUÇLARI ---")
print(f"ROC-AUC Skoru: {roc_auc_score(y_val, y_pred_proba):.4f}")
print(f"F1-Score: {f1_score(y_val, y_pred):.4f}")
print("\nDetaylı Sınıflandırma Raporu:\n", classification_report(y_val, y_pred))

Eğitim Seti Boyutu: (553095, 22)
Doğrulama (Val) Seti Boyutu: (138274, 22)
[LightGBM] [Info] Number of positive: 392379, number of negative: 160716
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.071289 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3234
[LightGBM] [Info] Number of data points in the train set: 553095, number of used features: 22
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.709424 -> initscore=0.892589
[LightGBM] [Info] Start training from score 0.892589

--- MODEL PERFORMANS SONUÇLARI ---
ROC-AUC Skoru: 0.9535
F1-Score: 0.9226

Detaylı Sınıflandırma Raporu:
               precision    recall  f1-score   support

           0       0.83      0.77      0.80     40179
           1       0.91      0.94      0.92     98095

    accuracy                           0.89    138274
   macro avg       0.87      0.85      0.86    138274
weighted avg       0.89      0.89      0.89    1382

In [11]:
import xgboost as xgb

# 1. XGBoost Modelini Tanımlama
model_xgb = xgb.XGBClassifier(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    random_state=42,
    n_jobs=-1,
    eval_metric='logloss'
)

# 2. Modeli Eğitme (Early Stopping ile)
model_xgb.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=False
)

# 3. Tahmin Alma
y_pred_proba_xgb = model_xgb.predict_proba(X_val)[:, 1]
y_pred_xgb = model_xgb.predict(X_val)

# 4. Performans Metrikleri
print("\n--- XGBOOST MODEL PERFORMANS SONUÇLARI ---")
print(f"XGBoost ROC-AUC Skoru: {roc_auc_score(y_val, y_pred_proba_xgb):.4f}")
print(f"XGBoost F1-Score: {f1_score(y_val, y_pred_xgb):.4f}")


--- XGBOOST MODEL PERFORMANS SONUÇLARI ---
XGBoost ROC-AUC Skoru: 0.9519
XGBoost F1-Score: 0.9212


In [12]:
from catboost import CatBoostClassifier

# 1. CatBoost Modelini Tanımlama
model_cat = CatBoostClassifier(
    iterations=500,
    learning_rate=0.05,
    depth=6,
    random_state=42,
    verbose=False
)

# 2. Modeli Eğitme
model_cat.fit(X_train, y_train, eval_set=(X_val, y_val), early_stopping_rounds=50)

# 3. Tahmin Alma
y_pred_proba_cat = model_cat.predict_proba(X_val)[:, 1]
y_pred_cat = model_cat.predict(X_val)

# 4. Performans Metrikleri
print("\n--- CATBOOST MODEL PERFORMANS SONUÇLARI ---")
print(f"CatBoost ROC-AUC Skoru: {roc_auc_score(y_val, y_pred_proba_cat):.4f}")
print(f"CatBoost F1-Score: {f1_score(y_val, y_pred_cat):.4f}")


--- CATBOOST MODEL PERFORMANS SONUÇLARI ---
CatBoost ROC-AUC Skoru: 0.9450
CatBoost F1-Score: 0.9158


In [13]:
import os
import pandas as pd

# 1. Test Dosyası Yolunu Otomatik Bulma
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        if 'test' in filename:
            test_path = os.path.join(dirname, filename)

test_df = pd.read_csv(test_path)

# 2. Test Verisini Ön İşleme ve X_train Sütunlarıyla Birebir Eşitleme
# Categorical verileri sayısal formata çeviriyoruz (One-Hot Encoding)
test_processed = pd.get_dummies(test_df)

# X_train'de olup X_test'te olmayan eksik sütunları 0 ile dolduruyoruz
for col in X_train.columns:
    if col not in test_processed.columns:
        test_processed[col] = 0

# Sütun sırasını X_train ile birebir aynı yapıyoruz
X_test = test_processed[X_train.columns]

# 3. Modellerden Test Tahminlerini Alma
test_preds_lgb = model.predict_proba(X_test)[:, 1]
test_preds_xgb = model_xgb.predict_proba(X_test)[:, 1]
test_preds_cat = model_cat.predict_proba(X_test)[:, 1]

# 4. Harmanlama (Ensemble) ve submission.csv Oluşturma
final_test_preds = (0.50 * test_preds_lgb) + (0.35 * test_preds_xgb) + (0.15 * test_preds_cat)

submission = pd.DataFrame({
    'id': test_df['id'],
    'addicted_label': final_test_preds
})

submission.to_csv('submission.csv', index=False)
print("Tebrikler ! 'submission.csv' başarıyla oluşturuldu!")

Tebrikler ! 'submission.csv' başarıyla oluşturuldu!


In [14]:
# ==========================================
# 1. ADVANCED FEATURE ENGINEERING (Yeni Özellikler)
# ==========================================
def create_advanced_features(df):
    df_feat = df.copy()
    
    if 'Daily_Screen_Time_Hours' in df_feat.columns and 'Age' in df_feat.columns:
        df_feat['screen_time_per_age'] = df_feat['Daily_Screen_Time_Hours'] / (df_feat['Age'] + 1)
        
    if 'Daily_Screen_Time_Hours' in df_feat.columns and 'App_Usage_Time_Hours' in df_feat.columns:
        df_feat['app_to_screen_ratio'] = df_feat['App_Usage_Time_Hours'] / (df_feat['Daily_Screen_Time_Hours'] + 0.01)
        
    if 'Social_Media_Time_Hours' in df_feat.columns and 'Daily_Screen_Time_Hours' in df_feat.columns:
        df_feat['social_ratio'] = df_feat['Social_Media_Time_Hours'] / (df_feat['Daily_Screen_Time_Hours'] + 0.01)

    if 'Sleep_Duration_Hours' in df_feat.columns and 'Daily_Screen_Time_Hours' in df_feat.columns:
        df_feat['screen_sleep_diff'] = df_feat['Daily_Screen_Time_Hours'] - df_feat['Sleep_Duration_Hours']

    return df_feat

# Yeni verileri oluşturma ve hizalama
X_train_adv = create_advanced_features(X_train)
X_test_adv = create_advanced_features(test_processed)
X_train_adv, X_test_adv = X_train_adv.align(X_test_adv, join='left', axis=1, fill_value=0)

# ==========================================
# 2. MODELLERİ YENİ ÖZELLİKLERLE EĞİTME
# ==========================================
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from sklearn.linear_model import LogisticRegression

# Modellerin Tanımlanması
lgb = LGBMClassifier(random_state=42, verbose=-1)
xgb = XGBClassifier(random_state=42, eval_metric='logloss')
cat = CatBoostClassifier(random_state=42, verbose=0)

# Eğitme
lgb.fit(X_train_adv, y_train)
xgb.fit(X_train_adv, y_train)
cat.fit(X_train_adv, y_train)

# ==========================================
# 3. STACKING ENSEMBLE (Meta-Learner)
# ==========================================
# 1. Seviye Tahminleri
oof_lgb = lgb.predict_proba(X_train_adv)[:, 1]
oof_xgb = xgb.predict_proba(X_train_adv)[:, 1]
oof_cat = cat.predict_proba(X_train_adv)[:, 1]

test_lgb = lgb.predict_proba(X_test_adv)[:, 1]
test_xgb = xgb.predict_proba(X_test_adv)[:, 1]
test_cat = cat.predict_proba(X_test_adv)[:, 1]

S_train = np.column_stack([oof_lgb, oof_xgb, oof_cat])
S_test = np.column_stack([test_lgb, test_xgb, test_cat])

# Meta Model
meta_model = LogisticRegression()
meta_model.fit(S_train, y_train)

final_preds = meta_model.predict_proba(S_test)[:, 1]

# ==========================================
# 4. KONTROL VE KAYDETME
# ==========================================
# İlk submission dosyasını temel alarak boyut/format kontrolü yapıyoruz
submission = pd.read_csv('submission.csv') if 'submission' not in locals() else submission
submission['addicted_label'] = final_preds
submission.to_csv('submission.csv', index=False)

print("Gelişmiş Stacking modeli başarıyla eğitildi ve submission.csv güncellendi!")

Gelişmiş Stacking modeli başarıyla eğitildi ve submission.csv güncellendi!
